In [1]:
import pandas as pd
import plotly.express as px

def prepare_time_column(df):
    df = df.copy()
    df["time"] = pd.to_datetime(df["time"])
    return df

def plot_time_series(df, title, ylabel="Value", percent=False):
    df = prepare_time_column(df)

    fig = px.line(
        df.sort_values("time"),
        x="time",
        y="value",
        color="metric",
        markers=True,
        title=title,
        labels={"time": "Time", "value": ylabel, "metric": "Metric"},
    )

    if percent:
        fig.update_yaxes(tickformat=".0%")

    # Mark every Friday and Monday in the time range with vertical lines.
    tmin, tmax = df["time"].min(), df["time"].max()
    if pd.notna(tmin) and pd.notna(tmax):
        days = pd.date_range(tmin.normalize(), tmax.normalize(), freq="D")
        weekday_styles = {
            0: ("Mon", "#1f77b4"),  # Monday
            4: ("Fri", "#d62728"),  # Friday
        }
        seen_labels = set()
        for d in days:
            if d.weekday() not in weekday_styles:
                continue
            label, color = weekday_styles[d.weekday()]
            show_legend = label not in seen_labels
            seen_labels.add(label)
            fig.add_vline(
                x=d,
                line_width=1,
                line_dash="dot",
                line_color=color,
                opacity=0.5,
            )
            # Invisible scatter trace just to get a legend entry per weekday.
            if show_legend:
                fig.add_scatter(
                    x=[d],
                    y=[None],
                    mode="lines",
                    line=dict(color=color, dash="dot"),
                    name=label,
                    showlegend=True,
                    hoverinfo="skip",
                )

    fig.update_layout(
        template="plotly_white",
        hovermode="x unified",
        legend_title_text="",
        width=1000,
        height=500,
    )

    fig.show()


In [2]:
from python_utilities.db_connection import DbConnection 
analytics_db = DbConnection('ANALYTICS', 'PROD_RDS')

INFO [2026-05-08 10:08:25] - PYTHON_UTILITIES - secret_utilities.py - get_db_secret_config - Credentials for database were read from secret.ini file


In [3]:
start_date = "2026-05-01"
end_date = "2026-05-10"
time_bucket = "day" # or hour


In [4]:
bucket_expr = {
    "day": "DATE(created_at)",
    "hour": "DATE_FORMAT(created_at, '%%Y-%%m-%%d %%H:00:00')",
}[time_bucket]


# Tracking Aftercourt Binary Model Predictions

### Prediction Volume per Model Over Time

In [5]:
df = analytics_db.sql_to_df(f"""
SELECT
  {bucket_expr} AS time,
  model_name AS metric,
  COUNT(*) AS value
FROM llm_attachments_predictions
WHERE
  created_at >= '{start_date}'
  AND created_at < '{end_date}'
  AND model_name IN (
    'aftercourt_classification_ladung',
    'aftercourt_classification_pfub',
    'pfub_erlass_egvp'
  )
GROUP BY 1, model_name
ORDER BY 1;
""")

In [6]:
df

,time,metric,value
0,2026-05-01,aftercourt_classification_ladung,3177
1,2026-05-01,aftercourt_classification_pfub,3168
2,2026-05-01,pfub_erlass_egvp,79
3,2026-05-02,aftercourt_classification_ladung,5225
4,2026-05-02,aftercourt_classification_pfub,5190
5,2026-05-02,pfub_erlass_egvp,105
6,2026-05-03,aftercourt_classification_ladung,3258
7,2026-05-03,aftercourt_classification_pfub,3195
8,2026-05-03,pfub_erlass_egvp,146
9,2026-05-04,aftercourt_classification_ladung,16993


In [7]:
plot_time_series(df, title="LLM Attachment Predictions per Model", ylabel="Count")

### Unique Attachment Processed Per Model

In [8]:
df = analytics_db.sql_to_df(f"""
SELECT
  {bucket_expr} AS time,
  model_name AS metric,
  COUNT(DISTINCT attachment_id) AS value
FROM llm_attachments_predictions
WHERE
  created_at >= '{start_date}'
  AND created_at < '{end_date}'
  AND model_name IN (
    'aftercourt_classification_ladung',
    'aftercourt_classification_pfub',
    'pfub_erlass_egvp'
  )
GROUP BY 1, model_name
ORDER BY 1;
""")

In [9]:
plot_time_series(df, title="LLM Attachment Predictions per Model (Unique Attachments)", ylabel="Count")

### Binary Classifier Positive Rate Over Time

In [10]:
df = analytics_db.sql_to_df(f"""
SELECT
  {bucket_expr} AS time,
  model_name AS metric,
  AVG(
    CASE
      WHEN LOWER(value) LIKE '%%true%%' THEN 1.0
      WHEN LOWER(value) LIKE '%%false%%' THEN 0.0
      ELSE NULL
    END
  ) AS value
FROM llm_attachments_predictions
WHERE
  created_at >= '{start_date}'
  AND created_at < '{end_date}'
  AND model_name IN (
    'aftercourt_classification_ladung',
    'aftercourt_classification_pfub'
  )
  AND subtype = 'class_pred'
GROUP BY 1, model_name
ORDER BY 1;
""")

In [11]:
plot_time_series(df, title="Aftercourt Binary Classifiers Positive Prediction Rate", ylabel="Positive Rate")

### Binary Class Positive Count Over Time

In [12]:
print(start_date)

2026-05-01


In [13]:
print(end_date)

2026-05-10


In [14]:
bucket_expr_hour = {
    "day": "DATE(created_at)",
    "hour": "DATE_FORMAT(created_at, '%%Y-%%m-%%d %%H:00:00')",
}['hour']
df = analytics_db.sql_to_df(f"""
SELECT
  {bucket_expr_hour} AS time,
  model_name AS metric,
  SUM(CASE WHEN LOWER(value) LIKE '%%true%%' THEN 1 ELSE 0 END) AS value
FROM llm_attachments_predictions
WHERE
  created_at >= '{start_date}'
  AND created_at < '{end_date}'
  AND model_name IN (
    'aftercourt_classification_ladung',
    'aftercourt_classification_pfub'
  )
  AND subtype = 'class_pred'
GROUP BY 1, model_name
ORDER BY 1;
""")

In [15]:
plot_time_series(df, title="Aftercourt Binary Classifiers Positive Prediction Rate", ylabel="Positive Count")

### Binary Classifier Class Counts Over Time

In [16]:
df = analytics_db.sql_to_df(f"""
SELECT
  {bucket_expr} AS time,
  CONCAT(model_name, ':', LOWER(TRIM(value))) AS metric,
  COUNT(*) AS value
FROM llm_attachments_predictions
WHERE
  created_at >= '{start_date}'
  AND created_at < '{end_date}'
  AND model_name IN (
    'aftercourt_classification_ladung',
    'aftercourt_classification_pfub'
  )
  AND subtype = 'class_pred'
GROUP BY 1, metric
ORDER BY 1;
""")

In [17]:
plot_time_series(df, title="LLM Attachment Predictions per Model (Unique Attachments)", ylabel="Binary Classifier Class Counts")

### Average class probability over time

In [18]:
df = analytics_db.sql_to_df(f"""
SELECT
  {bucket_expr} AS time,
  model_name AS metric,
  AVG(CAST(REPLACE(value, '''', '') AS DECIMAL(10, 6))) AS value
FROM llm_attachments_predictions
WHERE
  created_at >= '{start_date}'
  AND created_at < '{end_date}'
  AND model_name IN (
    'aftercourt_classification_ladung',
    'aftercourt_classification_pfub'
  )
  AND subtype = 'class_prob'
GROUP BY 1, model_name
ORDER BY 1;
""")

In [19]:
plot_time_series(df, title="Avarage Class Probability Over Time per Class", ylabel="Average Class Probability")

### Probability buckets per classifier

In [20]:
df = analytics_db.sql_to_df(f"""
SELECT
  {bucket_expr} AS time,
  CONCAT(
    model_name,
    ':',
    CASE
      WHEN CAST(REPLACE(value, '''', '') AS DECIMAL(10, 6)) < 0.1 THEN '0.0-0.1'
      WHEN CAST(REPLACE(value, '''', '') AS DECIMAL(10, 6)) < 0.2 THEN '0.1-0.2'
      WHEN CAST(REPLACE(value, '''', '') AS DECIMAL(10, 6)) < 0.3 THEN '0.2-0.3'
      WHEN CAST(REPLACE(value, '''', '') AS DECIMAL(10, 6)) < 0.4 THEN '0.3-0.4'
      WHEN CAST(REPLACE(value, '''', '') AS DECIMAL(10, 6)) < 0.5 THEN '0.4-0.5'
      WHEN CAST(REPLACE(value, '''', '') AS DECIMAL(10, 6)) < 0.6 THEN '0.5-0.6'
      WHEN CAST(REPLACE(value, '''', '') AS DECIMAL(10, 6)) < 0.7 THEN '0.6-0.7'
      WHEN CAST(REPLACE(value, '''', '') AS DECIMAL(10, 6)) < 0.8 THEN '0.7-0.8'
      WHEN CAST(REPLACE(value, '''', '') AS DECIMAL(10, 6)) < 0.9 THEN '0.8-0.9'
      ELSE '0.9-1.0'
    END
  ) AS metric,
  COUNT(*) AS value
FROM llm_attachments_predictions
WHERE
  created_at >= '{start_date}'
  AND created_at < '{end_date}'
  AND model_name IN (
    'aftercourt_classification_ladung',
    'aftercourt_classification_pfub'
  )
  AND subtype = 'class_prob'
GROUP BY 1, metric
ORDER BY 1, metric;
""")

In [21]:
plot_time_series(df, title="Class Probability Bucket Counts per Model Over Time", ylabel="Count")

### Classifier probability by prediction result

In [22]:
pred_bucket_expr = bucket_expr.replace("created_at", "pred.created_at")

df = analytics_db.sql_to_df(f"""
WITH pred AS (
  SELECT
    attachment_id,
    model_name,
    created_at,
    CASE
      WHEN LOWER(value) LIKE '%%true%%' THEN 'true'
      WHEN LOWER(value) LIKE '%%false%%' THEN 'false'
      ELSE NULL
    END AS class_pred
  FROM llm_attachments_predictions
  WHERE
    model_name IN (
      'aftercourt_classification_ladung',
      'aftercourt_classification_pfub'
    )
    AND subtype = 'class_pred'
),
prob AS (
  SELECT
    attachment_id,
    model_name,
    CAST(REPLACE(value, '''', '') AS DECIMAL(10, 6)) AS class_prob
  FROM llm_attachments_predictions
  WHERE
    model_name IN (
      'aftercourt_classification_ladung',
      'aftercourt_classification_pfub'
    )
    AND subtype = 'class_prob'
)
SELECT
  {pred_bucket_expr} AS time,
  CONCAT(pred.model_name, ':pred_', pred.class_pred) AS metric,
  AVG(prob.class_prob) AS value
FROM pred
JOIN prob
  ON pred.attachment_id = prob.attachment_id
  AND pred.model_name = prob.model_name
WHERE
  pred.created_at >= '{start_date}'
  AND pred.created_at < '{end_date}'
  AND pred.class_pred IS NOT NULL
GROUP BY 1, metric
ORDER BY 1;
""")

In [23]:
plot_time_series(df, title="Average Classifier Probability by Prediction Result", ylabel="Average Probability")

In [24]:
pred_bucket_expr = bucket_expr.replace("created_at", "pred.created_at")

df = analytics_db.sql_to_df(f"""
WITH pred AS (
  SELECT
    attachment_id,
    model_name,
    created_at,
    CASE
      WHEN LOWER(value) LIKE '%%true%%' THEN 'true'
      WHEN LOWER(value) LIKE '%%false%%' THEN 'false'
      ELSE NULL
    END AS class_pred
  FROM llm_attachments_predictions
  WHERE
    model_name IN (
      'aftercourt_classification_ladung',
      'aftercourt_classification_pfub'
    )
    AND subtype = 'class_pred'
),
prob AS (
  SELECT
    attachment_id,
    model_name,
    CAST(REPLACE(value, '''', '') AS DECIMAL(10, 6)) AS class_prob
  FROM llm_attachments_predictions
  WHERE
    model_name IN (
      'aftercourt_classification_ladung',
      'aftercourt_classification_pfub'
    )
    AND subtype = 'class_prob'
    AND CAST(REPLACE(value, '''', '') AS DECIMAL(10, 6)) > 0
)
SELECT
  {{pred_bucket_expr}} AS time,
  CONCAT(pred.model_name, ':pred_', pred.class_pred) AS metric,
  AVG(prob.class_prob) AS value
FROM pred
JOIN prob
  ON pred.attachment_id = prob.attachment_id
  AND pred.model_name = prob.model_name
WHERE
  pred.created_at >= '{{start_date}}'
  AND pred.created_at < '{{end_date}}'
  AND pred.class_pred IS NOT NULL
  AND prob.class_prob > 0
GROUP BY 1, metric
ORDER BY 1;
""".format(pred_bucket_expr=pred_bucket_expr, start_date=start_date, end_date=end_date))

In [25]:
plot_time_series(df, title="Average Classifier Probability by Prediction Result", ylabel="Average Probability excluding Zero Probabilities")

### Daily Probability Distribution (Boxplot)

In [26]:
prob_df = analytics_db.sql_to_df(f"""
SELECT
  DATE(created_at) AS day,
  model_name,
  CAST(REPLACE(value, '''', '') AS DECIMAL(10, 6)) AS prob
FROM llm_attachments_predictions
WHERE
  created_at >= '{start_date}'
  AND created_at < '{end_date}'
  AND model_name IN (
    'aftercourt_classification_ladung',
    'aftercourt_classification_pfub'
  )
  AND subtype = 'class_prob';
""")
prob_df["day"] = pd.to_datetime(prob_df["day"]).dt.strftime("%Y-%m-%d")
prob_df["prob"] = prob_df["prob"].astype(float)

for model in ["aftercourt_classification_ladung", "aftercourt_classification_pfub"]:
    model_df = prob_df[prob_df["model_name"] == model].sort_values("day")
    fig = px.strip(
        model_df,
        x="day",
        y="prob",
        title=f"Daily Class Probability Distribution - {model}",
        labels={"day": "Day", "prob": "Class Probability"},
    )
    fig.update_traces(jitter=0.8, marker=dict(size=4, opacity=0.5))
    fig.update_layout(
        template="plotly_white",
        width=1100,
        height=550,
        legend_title_text="",
    )
    fig.update_yaxes(range=[0, 1])
    fig.show()

# What happened on a special day?

In [27]:
day = "2026-05-04"



## Continue analysis for pfub erlass